In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Synthetic Legacy Clinic Data Generator
# MAGIC This notebook fetches real data from the NPPES API and generates synthetic "dirty" legacy records for Master Data Management (MDM) testing.

# COMMAND ----------
import requests
import random
import string
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType
import re

# COMMAND ----------
# Defining the input widgets (Widget Name, Default Value, Widget Label)
# Note: Default state changed to California (CA)
dbutils.widgets.text("catalog", "bronze", "1. Unity Catalog Name")
dbutils.widgets.text("schema", "cms_mdm_project", "2. Schema/Database Name")
dbutils.widgets.text("table_name", "synthetic_clinic_provider_data", "3. Target Table Name")
dbutils.widgets.text("state", "CA", "4. Target State Filter (e.g., CA)")

# Retrieving the values from the widgets
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table_name = dbutils.widgets.get("table_name")
state = dbutils.widgets.get("state")

# COMMAND ----------
# MAGIC %md
# MAGIC **Troubleshooting Library Imports:**
# MAGIC If you run into any module not found errors when executing the import cell above (such as `requests` missing), simply add a new cell below this one and execute the following command to install the required libraries:
# MAGIC 
# MAGIC `%pip install requests` 
# MAGIC *(Note: If your cluster supports `uv`, you can also use `%pip install uv && uv pip install requests`)*

# COMMAND ----------
def introduce_typo(text):
    """Randomly drops or swaps a character in a string to simulate a typo."""
    if not text or len(text) < 3:
        return text
    
    action = random.choice(['drop', 'swap', 'none'])
    if action == 'none':
        return text
        
    idx = random.randint(0, len(text) - 2)
    text_list = list(text)
    
    if action == 'drop':
        text_list.pop(idx)
    elif action == 'swap':
        text_list[idx], text_list[idx+1] = text_list[idx+1], text_list[idx]
        
    return "".join(text_list)

def format_phone_badly(phone):
    """Takes a standard 10 digit phone and mangles the formatting."""
    # Strip out all non-digit characters
    phone = re.sub(r'\D', '', str(phone) if phone else '')

    if not phone or len(phone) != 10:
        return phone
    
    formats = [
        f"{phone[:3]}-{phone[3:6]}-{phone[6:]}",  # Standard dashes
        f"({phone[:3]}) {phone[3:6]}-{phone[6:]}", # Parentheses
        f"{phone[:3]}.{phone[3:6]}.{phone[6:]}",  # Dots
        f"1-{phone[:3]}-{phone[3:6]}-{phone[6:]}", # Country code
        phone # No formatting
    ]
    return random.choice(formats)

def random_past_date():
    """Generates a random date within the last 5 years to test Survivorship rules."""
    start_date = datetime.now() - timedelta(days=5*365)
    random_days = random.randint(0, 5*365)
    return (start_date + timedelta(days=random_days)).strftime('%Y-%m-%d')

# COMMAND ----------
def fetch_base_npi_data(state="CA", limit=50, taxonomy="Pediatrics"):
    """Fetches real provider data from the NPPES API to use as our base."""
    print(f"Fetching {limit} records from NPPES API for state {state}...")
    url = "https://npiregistry.cms.hhs.gov/api/"

    params = {
    "version": "2.1",
    "enumeration_type": "NPI-1",
    "state": state,
    "country_code": "US",
    "taxonomy_description": taxonomy,  # Added to prevent empty return
    "limit": limit,
    "pretty": "true"
        }
    
    response = requests.get(url, params=params)

    if response.status_code != 200:
        raise Exception(f"API Request failed with status {response.status_code}")

    results = response.json().get('results', [])

    print(f"API Request successful. Got {len(results)} records.")
        
    return results

# COMMAND ----------
def generate_legacy_dataset(api_results):
    """Takes clean API data and generates dirty legacy system records."""
    print("Generating dirty legacy records...")
    legacy_records = []
    legacy_id_counter = 10000
    
    for provider in api_results:
        basic_info = provider.get('basic', {})
        addresses = provider.get('addresses', [{}])
        primary_address = next((addr for addr in addresses if addr.get('address_purpose') == 'LOCATION'), addresses[0])
        
        # We will create 1 to 3 "legacy" records for each real provider to simulate fragmentation
        num_legacy_records = random.randint(1, 3)
        
        for _ in range(num_legacy_records):
            legacy_id_counter += 1
            
            # 1. Base clean extraction
            npi = provider.get('number')
            first_name = basic_info.get('first_name', '')
            last_name = basic_info.get('last_name', '')
            phone = primary_address.get('telephone_number', '')
            address = primary_address.get('address_1', '')
            city = primary_address.get('city', '')
            record_state = primary_address.get('state', '')
            zip_code = primary_address.get('postal_code', '')[:5]
            
            # 2. Intentional Corruptions
            if random.random() < 0.20:
                npi = None 
                
            if random.random() < 0.30:
                first_name = introduce_typo(first_name)
            if random.random() < 0.30:
                last_name = introduce_typo(last_name)
                
            phone = format_phone_badly(phone)
            
            if random.random() < 0.15:
                phone = None
            if random.random() < 0.15:
                address = None
                
            # Create a PySpark Row directly instead of a generic dict
            record = Row(
                Legacy_System_ID=f"SYS_A_{legacy_id_counter}",
                NPI=str(npi) if npi else None,
                First_Name=first_name,
                Last_Name=last_name,
                Practice_Address=address,
                City=city,
                State=record_state,
                Zip=zip_code,
                Phone=phone,
                Last_Updated_Date=random_past_date(),
                Source_System=random.choice(["Acquired_Clinic_Alpha", "Legacy_HR_DB", "Old_Billing_System"])
            )
            legacy_records.append(record)
            
    return legacy_records



In [0]:
# COMMAND ----------
# Fetch real records using the widget state parameter
real_data = fetch_base_npi_data(state=state, limit=50, taxonomy="Pediatrics")

In [0]:
# Generate the dirty list of PySpark Rows
dirty_rows = generate_legacy_dataset(real_data)

# Create a PySpark DataFrame directly from the list of Rows (No Pandas needed)
spark_df = spark.createDataFrame(dirty_rows)

In [0]:


# Construct the fully qualified table name for Unity Catalog
full_table_name = f"{catalog}.{schema}.{table_name}"
print(f"Writing dirty legacy data to Bronze table: {full_table_name}...")

# Write directly to a Delta table in Unity Catalog
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(full_table_name)

print(f"Success! Created Delta table '{full_table_name}' with {spark_df.count()} simulated dirty records.")

# Display the output directly in the notebook cell
display(spark_df)